# Error Analysis

This notebook inspects the final released taxonomy annotations and reproduces key table-style summaries.
It is intended for qualitative review and light analysis, not as the primary batch pipeline.

In [10]:
from pathlib import Path
import pandas as pd
import sys

def find_root(start: Path) -> Path:
    for path in (start.resolve(), *start.resolve().parents):
        if (path / "src").exists() and (path / "annotations").exists():
            return path
    raise FileNotFoundError("Could not locate repository root.")

ROOT = find_root(Path.cwd())
sys.path.insert(0, str(ROOT / "src"))

from groundlm.analysis.labels import aggregated_error_rates, label_rates_by_model

path = ROOT / "annotations" / "qualitative_direct.csv"
df = pd.read_csv(path)
df.head()


,model_name,sample_index,error_label,Review_Notes,reference,prediction,dialogue,rouge1,rouge2,rougeL,bs_f1_raw,informativeness,clarity,plausibility,faithfulness,Length Ratio
0,aya-expanse:8b,0,NO-ERROR,The summary accurately reflects the dialogue's...,佩顿给麦克斯提供了卖衣服的网站。他喜欢浏览、试穿衣服，但不一定都会买。他经常买衣服和书，因为...,麦克斯问派顿是否知道哪里可以买到好衣服。派顿列出了几个网站，但表示通常会从两个或三个网站购买...,Max: Know any good sites to buy clothes from?\...,34.72,12.68,26.39,72.35,3.120808,3.776156,3.844905,2.919892,3.301887
1,aya-expanse:8b,1,H-CIRC,"It reverses the quantity relation, saying ther...",凯利很害怕芬兰雕塑花园里的人物，因为她觉得那些人物的脸非常病态。对奥利来说，这就像日本的人偶...,奥利、凯利、米奇和杰西卡在聊天，分享他们认为最可怕或最令人印象深刻的地方。凯利提到芬兰的一个...,"Ollie: Okay, Kelly! Ur up nxt!\n Kelly: Me? I ...",24.00,3.25,16.00,66.78,3.343914,3.841215,3.990529,3.391250,2.709091
2,aya-expanse:8b,2,NO-ERROR,The summary accurately reflects the dialogue's...,贝拉和埃里克拒绝了一个客户的要求。他们老板很赞赏这个决定。他会吸引新的客户来。,埃里克问贝拉今天老板房间里发生了什么，老板是否生气。贝拉回答说老板并没有生气，反而对他们拒绝...,"Eric: Hey Bella, What happened today in boss's...",36.59,0.00,21.95,70.33,2.793783,3.780327,3.698617,2.799404,2.769231
3,aya-expanse:8b,3,H-CIRC|LANG,It invents a specific amount of several hundre...,玛丽的钱用完了，卡特打算一小时后借给她一点。,玛丽：嘿，我有点缺钱，能借我几百元吗？\n 卡特：好的，给我一个小时，我在火车站。\n 玛丽...,"Mary: hey, im kinda broke, lend me a few box\n...",20.83,4.35,16.67,68.02,4.053118,4.851738,4.813985,3.750223,2.090909
4,aya-expanse:8b,4,NO-ERROR,The summary accurately reflects the dialogue's...,杰克和梅一会儿要喝鸡尾酒。,杰克提议后来喝鸡尾酒，梅激动地回应“是！”她表示自己确实很紧张。杰克猜测了，梅叹了口气，承认...,Jack: Cocktails later?\n May: YES!!!\n May: Yo...,19.23,4.00,11.54,64.97,3.440671,3.597595,3.739751,2.979432,5.692308


In [11]:
aggregated_error_rates(df)


,model_name,no_error,hallucination,omission,language
0,aya-expanse:8b,0.38,0.43,0.07,0.17
1,gemma4:e4b,0.63,0.29,0.08,0.00
2,qwen3.5:9b,0.58,0.39,0.03,0.00
3,mBART-large,0.20,0.61,0.48,0.11


In [12]:
from groundlm.config import MODEL_ORDER, TAXONOMY_LABELS

label_long = df[["model_name", "error_label"]].copy()
label_long["label"] = (
    label_long["error_label"]
    .fillna("")
    .astype(str)
    .str.split(r"[|,;]+", regex=True)
)
label_long = label_long.explode("label")
label_long["label"] = label_long["label"].astype(str).str.strip()
label_long = label_long[label_long["label"] != ""]

counts = (
    label_long.groupby(["model_name", "label"]).size().unstack(fill_value=0)
    .reindex(index=MODEL_ORDER, columns=TAXONOMY_LABELS, fill_value=0)
)

fine_grained = counts.div(100)
fine_grained


label,H-ENT,H-EVT,H-ROLE,H-CIRC,H-MOD,H-DISC,OMIT,LANG,NO-ERROR
model_name,,,,,,,,,
aya-expanse:8b,0.06,0.05,0.21,0.10,0.03,0.03,0.07,0.17,0.38
gemma4:e4b,0.10,0.06,0.05,0.03,0.05,0.02,0.08,0.00,0.63
qwen3.5:9b,0.07,0.11,0.14,0.10,0.03,0.07,0.03,0.00,0.58
mBART-large,0.16,0.32,0.21,0.16,0.15,0.10,0.48,0.11,0.20


In [13]:
sample_index = 0
df[df["sample_index"] == sample_index][[
    "model_name",
    "error_label",
    "Review_Notes",
    "prediction",
    "reference",
]]


,model_name,error_label,Review_Notes,prediction,reference
0,aya-expanse:8b,NO-ERROR,The summary accurately reflects the dialogue's...,麦克斯问派顿是否知道哪里可以买到好衣服。派顿列出了几个网站，但表示通常会从两个或三个网站购买...,佩顿给麦克斯提供了卖衣服的网站。他喜欢浏览、试穿衣服，但不一定都会买。他经常买衣服和书，因为...
100,gemma4:e4b,H-MOD,"It reverses 'hardest to resist,' saying clothe...",两人讨论了购买衣服的网站，并分享了各自的购物习惯。Max询问Payton是否喜欢购物，Pay...,佩顿给麦克斯提供了卖衣服的网站。他喜欢浏览、试穿衣服，但不一定都会买。他经常买衣服和书，因为...
200,qwen3.5:9b,NO-ERROR,The summary accurately reflects the dialogue's...,Max 询问购买衣物的网站，Payton 推荐了多个并解释自己通常只从其中两三个选购。随后两...,佩顿给麦克斯提供了卖衣服的网站。他喜欢浏览、试穿衣服，但不一定都会买。他经常买衣服和书，因为...
300,mBART-large,H-EVT|H-CIRC|OMIT,It invents a visit to Payton's clothing store ...,马克斯会去佩顿的服装店买东西。,佩顿给麦克斯提供了卖衣服的网站。他喜欢浏览、试穿衣服，但不一定都会买。他经常买衣服和书，因为...
